In [ ]:
# 기초적인 Scaled Dot-Product Attention 예제 (Tokenizer + Attention 포함)
import numpy as np
import math
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# 인코딩(Encoding) 부분
# 1. 입력 / 출력 문장 정의
input_text = "I have a pen"
output_text = "<sos> 나는 펜을 갖고 있다 <eos>"

# 2. 토크나이저 설정 및 시퀀스 변환
input_tokenizer = Tokenizer(filters='', lower=False)
# filters='': 특수문자 제거하지 않음 (기본값은 마침표, 쉼표 등을 제거함)

output_tokenizer = Tokenizer(filters='', lower=False)

# 숫자 시퀀스는 딥러닝 모델에서 문장을 처리할 수 있도록 만들어주는 필수 전처리 단계
# 입력 문장 "I have a pen"을 학습시켜 단어 사전(word_index) 생성
# 예) { 'I': 1, 'have': 2, 'a': 3, 'pen': 4 } ← 숫자는 자동으로 할당됨
input_tokenizer.fit_on_texts([input_text])

# 출력 문장 "<sos> 나는 펜을 갖고 있다 <eos>"를 학습시켜 또 다른 단어 사전 생성
# 예) { '<sos>': 1, '나는': 2, '펜을': 3, '갖고': 4, '있다': 5, '<eos>': 6 }
output_tokenizer.fit_on_texts([output_text])

# Tokenizer를 이용해 아래처럼 문장을 숫자 시퀀스로 변환
input_seq = input_tokenizer.texts_to_sequences([input_text])     # ex: [[1,2,3,4]]
output_seq = output_tokenizer.texts_to_sequences([output_text])  # ex: [[1,2,3,4,5,6]]

# 3. 시퀀스 패딩 : 원문 문장을 단어 인덱스 시퀀스로 바꾸고, 고정 길이(=max_input_len)로 패딩.
# 이렇게 하면 데이터가 길어지거나 짧아져도, 항상 가장 긴 시퀀스에 맞춰 패딩 길이가 결정되므로
# 하드코딩 없이 유연하게 동작.
max_input_len  = max(len(s) for s in input_seq)   # 현재는 4
max_output_len = max(len(s) for s in output_seq)  # 현재는 6
input_pad = pad_sequences(input_seq, maxlen=max_input_len, padding='post')
output_pad = pad_sequences(output_seq, maxlen=max_output_len, padding='post')

# 4. Key, Value, Query 구성
n_src = input_pad.shape[1]   # 입력 길이 (패딩 포함)
n_tgt = output_pad.shape[1]  # 출력 길이 (패딩 포함)

# 여기서는 인코더 LSTM 없이 one-hot 벡터로 단어 특징을 단순화
# 실제 모델에서는 이 부분이 인코더(Embedding → LSTM → hidden states) 에 해당.
K = np.eye(n_src)   # 각 입력 단어를 one-hot vector로 간주 (Key)
V = np.eye(n_src)   # 같은 방식으로 Value도 동일하게 설정
# 왜 K, V에 np.eye()를 쓰는가?
# Key(K) 와 Value(V)는 보통 인코더에서 나온 벡터다.
# 하지만 이 예제는 복잡한 신경망을 포함하지 않기 때문에 K와 V를 그냥 one-hot 벡터로 대체.
# 즉, "I"는 [1, 0, 0, 0], "have"는 [0, 1, 0, 0]처럼 표현됨.
# 참고로 이 예제는 실제 번역 시스템이 아니라 "Attention의 원리만 설명하기 위한 도식 실험"용이다.
# 그래서 입력 단어 각각에 대한 고유 벡터를 복잡하게 훈련하는 대신 "그냥 눈에 보이는 one-hot 벡터로 하자"고 설정한 것

# 디코딩(Decoding) 부분 -----
# Query 설정 (위치 기반 단순 집중)
# 현재 예제에서는 연습이니까 Q를 하드코딩해서 어디에 집중할지 사람이 지정함
Q = np.zeros((n_tgt, n_src))
for i in range(n_tgt):
    if i == 0:
        Q[i, 0] = 1.0
    elif i == n_tgt - 1:
        Q[i, -1] = 1.0
    elif i < n_src - 1:
        Q[i, i:i+2] = 0.5
    else:
        Q[i, -1] = 1.0

# 5. 어텐션 함수 : Attention을 시뮬레이션하는 함수
def attentionFunc(q, K, V):
    scores = q.dot(K.T) / math.sqrt(K.shape[1]) # 유사도 계산   Q•Kᵀ / √dₖ
    exp = np.exp(scores - np.max(scores))
    weights = exp / exp.sum()     # softmax 사용. 가중치 = 집중 정도
    context = (weights[:, None] * V).sum(axis=0)  # 가중합 → 컨텍스트
    return context, weights       # 집중된 정보 합산
    # weights가 실제로 각 입력 단어에 대해 디코더가 얼마나 주목했는지를 나타낸다.

# 6. Attention 실행 (디코더가 한 스텝씩 생성하는 과정)
print("\nAttention 결과 =====\n")
for i in range(n_tgt):
    context, weights = attentionFunc(Q[i], K, V)
    # 역할: 각 디코더 스텝마다 Query(q)를 K, V에 적용해 어디에 집중하고, 그 정보를 합친 컨텍스트 벡터를 얻음.
    # 출력 위치 i에 대한 집중 가중치(weights)와 컨텍스트(context) 확인
    print(f"[Output 위치 {i}]")

    for src_word, w in zip(input_tokenizer.word_index.keys(), weights):
        print(f"  - {src_word:>5} → Attention: {w:.3f}")
    print("  -> Context 벡터:", np.round(context, 3), "\n")
    # [Output 위치 1]
    #   -   I → Attention: 0.018
    #   - have → Attention: 0.443
    #   -   a → Attention: 0.443
    #   - pen → Attention: 0.094
    #   → 출력의 두 번째 단어는 "have"와 "a"에 집중하고 있다는 뜻~~~.

# 7. 최종 출력 시퀀스 확인
print("입력 시퀀스:", input_pad)
print("출력 시퀀스:", output_pad)

# 8. 디코더 출력 복원 (불필요한 토큰 제거: 0, <sos>, <eos>)
# 인덱스 → 단어 매핑을 빠르게 할 수 있도록 output_tokenizer.word_index를 뒤집은(역으로 만든) 사전을 만드는 역할
# output_tokenizer.word_index  {'<sos>':1, '나는':2, '펜을':3, '갖고':4, '있다':5, '<eos>':6}
# 이를 {인덱스: 단어}로 뒤집으면 reverse_output_index {1:'<sos>', 2:'나는', 3:'펜을', 4:'갖고', 5:'있다', 6:'<eos>'}
reverse_output_index = {v: k for k, v in output_tokenizer.word_index.items()}
reconstructed = []

# 디코더가 예측한 정수 시퀀스(예: [1,2,3,4,5,6,0,0])를 순회할 때,
for idx in output_pad[0]:
    if idx == 0:
        continue  # 패딩 무시
    word = reverse_output_index.get(idx, "?")  # 정수 인덱스를 실제 단어로 바로 변환
    if word in ["<sos>", "<eos>"]:
        continue  # 시작/종료 토큰 제거
        # 결과적으로 패딩(0), <sos>, <eos> 등을 걸러내고 최종 번역문만 깔끔히 복원할 수 있다.
    reconstructed.append(word)  # Attention으로 얻은 컨텍스트 기반 출력 인덱스를 실제 단어로 바꿔 문장 생성.

print("\n최종 번역문 복원 =====")
print(" ".join(reconstructed))

# 인코딩: 입력 문장을 Key/Value 벡터로 변환하고(Tokenizer → pad_sequences → K, V 생성)
# 디코딩: Query를 정하고, Attention으로 Context를 뽑아내며, 그 컨텍스트를 기반으로 번역 문장을 복원하는 과정
# 이 구조가 Attention의 기본 흐름을 잘 시뮬레이션하고 있다.


Attention 결과 =====

[Output 위치 0]
  -     I → Attention: 0.355
  -  have → Attention: 0.215
  -     a → Attention: 0.215
  -   pen → Attention: 0.215
  -> Context 벡터: [0.355 0.215 0.215 0.215] 

[Output 위치 1]
  -     I → Attention: 0.219
  -  have → Attention: 0.281
  -     a → Attention: 0.281
  -   pen → Attention: 0.219
  -> Context 벡터: [0.219 0.281 0.281 0.219] 

[Output 위치 2]
  -     I → Attention: 0.219
  -  have → Attention: 0.219
  -     a → Attention: 0.281
  -   pen → Attention: 0.281
  -> Context 벡터: [0.219 0.219 0.281 0.281] 

[Output 위치 3]
  -     I → Attention: 0.215
  -  have → Attention: 0.215
  -     a → Attention: 0.215
  -   pen → Attention: 0.355
  -> Context 벡터: [0.215 0.215 0.215 0.355] 

[Output 위치 4]
  -     I → Attention: 0.215
  -  have → Attention: 0.215
  -     a → Attention: 0.215
  -   pen → Attention: 0.355
  -> Context 벡터: [0.215 0.215 0.215 0.355] 

[Output 위치 5]
  -     I → Attention: 0.215
  -  have → Attention: 0.215
  -     a → Attention: 0.215
  -